# 🩺 Fine-tuning médical (QLoRA) — mission expérimentale

Fine-tuning **LoRA 4-bit (QLoRA)** d'un modèle de base sur le dataset médical
[`ruslanmv/ai-medical-chatbot`](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot).

**À lancer sur Google Colab (GPU T4/A100).** Runtime → Modifier le type d'exécution → GPU.

⚠️ **Modèle EXPÉRIMENTAL** — ne pas déployer en production (cf. `medical_project/Readme.md`).
Ce notebook n'a **rien à voir** avec l'assistant financier compromis ; il part d'une base saine.

À la fin : remplir dans `tests_modele_financier.md` le **lien Colab + métriques** (loss, epochs).

In [ ]:
# 1) Dépendances
!pip -q install "transformers>=4.45" "peft>=0.12" "accelerate>=0.34" "bitsandbytes>=0.43" "datasets>=2.20" trl matplotlib

In [ ]:
# 2) Vérif GPU
import torch
print('CUDA dispo :', torch.cuda.is_available())
print('GPU       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (activer le GPU !)')

In [ ]:
# 3) Config
BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'   # base saine
MAX_SAMPLES = 3000     # POC : monter pour de meilleurs résultats
MAX_LEN     = 512
EPOCHS      = 3
OUTPUT_DIR  = 'phi35_medical_lora'

In [ ]:
# 4) Dataset médical : téléchargement + mise au format instruction/response + pseudo-anonymisation
import re
from datasets import load_dataset, Dataset

EMAIL = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')
PHONE = re.compile(r'(\+?\d[\d\s().-]{7,}\d)')

def anon(t):
    t = EMAIL.sub('[EMAIL]', t or '')
    t = PHONE.sub('[PHONE]', t)
    return t.strip()

raw = load_dataset('ruslanmv/ai-medical-chatbot', split='train')
print('Lignes brutes :', len(raw), '| colonnes :', raw.column_names)

rows, seen = [], set()
for r in raw:
    q, a = anon(r.get('Patient')), anon(r.get('Doctor'))
    if len(q) < 10 or len(a) < 20:
        continue
    sig = (q + '||' + a)[:300]
    if sig in seen:
        continue
    seen.add(sig)
    rows.append({'instruction': q, 'output': a[:4000]})
    if len(rows) >= MAX_SAMPLES:
        break
print('Exemples retenus :', len(rows))
ds = Dataset.from_list(rows)

In [ ]:
# 5) Tokenizer + modèle de base en 4-bit (QLoRA)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'right'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type=TaskType.CAUSAL_LM,
                  target_modules=['qkv_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 6) Mise en forme (template Phi-3) + tokenisation
def to_text(ex):
    return {'text': f"<|user|>\n{ex['instruction']}<|end|>\n<|assistant|>\n{ex['output']}<|end|>"}

def tokenize(ex):
    out = tok(ex['text'], truncation=True, padding='max_length', max_length=MAX_LEN)
    out['labels'] = out['input_ids'].copy()
    return out

ds_txt = ds.map(to_text)
ds_tok = ds_txt.map(tokenize, remove_columns=ds_txt.column_names)
print(ds_tok)

In [ ]:
# 7) Entraînement (Trainer + log de la loss)
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to='none',
    remove_unused_columns=False,
)
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=ds_tok, data_collator=collator)
trainer.train()

In [ ]:
# 8) Métriques : courbe de loss + résumé (à reporter dans tests_modele_financier.md)
import matplotlib.pyplot as plt
hist = [h for h in trainer.state.log_history if 'loss' in h]
steps = [h['step'] for h in hist]
loss  = [h['loss'] for h in hist]
plt.figure(figsize=(8,4)); plt.plot(steps, loss); plt.xlabel('step'); plt.ylabel('loss')
plt.title('Training loss — fine-tuning médical'); plt.grid(True); plt.show()
print('Loss finale :', loss[-1] if loss else 'n/a')
print('Epochs      :', EPOCHS, '| steps :', steps[-1] if steps else 'n/a')
print('Exemples    :', len(ds_tok))

In [ ]:
# 9) Test conversationnel rapide
def chat(q, max_new=200):
    p = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    ids = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=True, top_p=0.9,
                         temperature=0.7, repetition_penalty=1.1, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

for q in ['What are common symptoms of dehydration?',
          'What should I do for a mild fever at home?']:
    print('Q:', q); print('A:', chat(q), '\n', '-'*60)

In [ ]:
# 10) Sauvegarde de l'adaptateur LoRA
model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print('Adaptateur sauvegardé dans', OUTPUT_DIR)
# Option : !zip -r phi35_medical_lora.zip phi35_medical_lora  puis télécharger depuis Colab

## ✅ À reporter dans le rendu
- **Lien Colab** (Partager → Toute personne avec le lien).
- **Loss finale**, **nombre d'epochs**, **durée**, **nb d'exemples** (cellule 8).
- Capture de la **courbe de loss**.

## ⚠️ Avertissements (cf. medical_project/Readme.md)
- Un modèle fine-tuné ne remplace **jamais** un professionnel de santé.
- Validation par des experts obligatoire avant tout usage clinique.
- Données pseudo-anonymisées (RGPD). Modèle **expérimental**, non déployé.